# 高校数学とR Day 2【R版】

- 城北中学校・高等学校　中学3年・高校1年
- 夏期講習会III 2026/8/24~2026/8/28
- 担当:清水団

## 本日のテーマ:整数問題をプログラムで考えよう!

Julia版と同じ内容を **R** で体験するバージョンです。言語ごとの書き方の違いにも注目してみてください。

### 5日間の学習予定

- **Day 1**:基本計算・for文・if文 ✅
- **Day 2**:整数問題をプログラムで考えよう ← 今日
- **Day 3**:場合の数・組合せを実験しよう
- **Day 4**:関数・グラフ・最大最小
- **Day 5**:確率・シミュレーション


## 約数を全部求めよう

In [23]:
divisors <- function(n) (1:n)[n %% (1:n) == 0]

divisors(24)

[1]  1  2  3  4  6  8 12 24

In [24]:
c(length(divisors(24)), sum(divisors(24)))  # 約数の個数と和

[1]  8 60

## 素数判定

In [25]:
# 約数がちょうど2個なら素数
is_prime <- function(n) n >= 2 && length(divisors(n)) == 2

c(is_prime(17), is_prime(21))

[1]  TRUE FALSE

In [26]:
# 1〜30の素数
for (n in 1:30) {
    if (is_prime(n)) print(n)
}

[1] 2
[1] 3
[1] 5
[1] 7
[1] 11
[1] 13
[1] 17
[1] 19
[1] 23
[1] 29


## エラトステネスの篩(ふるい)

In [27]:
eratosthenes <- function(n) {
    is_p <- rep(TRUE, n)
    is_p[1] <- FALSE
    for (p in 2:floor(sqrt(n))) {
        if (is_p[p]) {
            is_p[seq(p*p, n, by = p)] <- FALSE
        }
    }
    (1:n)[is_p]
}

eratosthenes(100)

[1]  2  3  5  7 11 13 17 19 23 29 31 37 41 43 47 53 59 61 67 71 73 79 83 89 97

In [28]:
# 素数の割合の実験
for (n in c(100, 1000, 10000, 100000)) {
    k <- length(eratosthenes(n))
    cat(sprintf("%d 以下の素数: %d 個(割合 %.2f%%)\n", n, k, k/n*100))
}

100 以下の素数: 25 個(割合 25.00%)
1000 以下の素数: 168 個(割合 16.80%)
10000 以下の素数: 1229 個(割合 12.29%)
100000 以下の素数: 9592 個(割合 9.59%)


## 最大公約数とユークリッドの互除法

Rには組み込みのgcdがないので、**自分で作ります**(それが勉強!)。

In [29]:
my_gcd <- function(a, b) {
    while (b != 0) {
        r <- a %% b
        a <- b
        b <- r
    }
    a
}

my_gcd(1071, 1029)

[1] 21

In [30]:
# 最小公倍数は gcd から作れる
my_lcm <- function(a, b) a * b / my_gcd(a, b)
c(my_lcm(12, 18), my_gcd(12, 18) * my_lcm(12, 18) == 12 * 18)

[1] 36  1

## 完全数を探そう

In [31]:
proper_sum <- function(n) sum(divisors(n)) - n

for (n in 2:10000) {
    if (proper_sum(n) == n) {
        cat(n, "は完全数! 約数:", divisors(n), "\n")
    }
}

6 は完全数! 約数: 1 2 3 6 
28 は完全数! 約数: 1 2 4 7 14 28 
496 は完全数! 約数: 1 2 4 8 16 31 62 124 248 496 
8128 は完全数! 約数: 1 2 4 8 16 32 64 127 254 508 1016 2032 4064 8128 


## 双子素数を探そう

In [32]:
for (n in 2:100) {
    if (is_prime(n) && is_prime(n + 2)) {
        cat(sprintf("(%d, %d)\n", n, n + 2))
    }
}

(3, 5)
(5, 7)
(11, 13)
(17, 19)
(29, 31)
(41, 43)
(59, 61)
(71, 73)


## フェルマー数

大きな整数には **gmp** パッケージを使います(Juliaの big() に相当)。

In [33]:
system("apt-get install -y libgmp-dev") # Install system dependency for gmp
install.packages("gmp")
library(gmp)

for (n in 0:4) {
    F <- as.bigz(2)^(2^n) + 1
    cat(sprintf("F_%d = %s → 素数? %s\n", n, as.character(F), isprime(F) > 0))
}

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



F_0 = 3 → 素数? TRUE
F_1 = 5 → 素数? TRUE
F_2 = 17 → 素数? TRUE
F_3 = 257 → 素数? TRUE
F_4 = 65537 → 素数? TRUE


In [34]:
# F₅の素因数分解(オイラーの発見を1秒で!)
factorize(as.bigz(2)^32 + 1)

Big Integer ('bigz') object of length 2:
[1] 641     6700417

## ピタゴラス数を探そう

In [35]:
for (c_ in 1:50) for (b in 1:c_) for (a in 1:b) {
    if (a^2 + b^2 == c_^2) cat(sprintf("(%d, %d, %d)\n", a, b, c_))
}

(3, 4, 5)
(6, 8, 10)
(5, 12, 13)
(9, 12, 15)
(8, 15, 17)
(12, 16, 20)
(15, 20, 25)
(7, 24, 25)
(10, 24, 26)
(20, 21, 29)
(18, 24, 30)
(16, 30, 34)
(21, 28, 35)
(12, 35, 37)
(15, 36, 39)
(24, 32, 40)
(9, 40, 41)
(27, 36, 45)
(30, 40, 50)
(14, 48, 50)


## コラッツ予想:世界一有名な未解決問題

In [36]:
collatz_steps <- function(n) {
    steps <- 0
    while (n != 1) {
        n <- if (n %% 2 == 0) n %/% 2 else 3*n + 1
        steps <- steps + 1
    }
    steps
}

# 1〜30のステップ数
for (n in 1:30) cat(sprintf("%d → %d ステップ\n", n, collatz_steps(n)))

1 → 0 ステップ
2 → 1 ステップ
3 → 7 ステップ
4 → 2 ステップ
5 → 5 ステップ
6 → 8 ステップ
7 → 16 ステップ
8 → 3 ステップ
9 → 19 ステップ
10 → 6 ステップ
11 → 14 ステップ
12 → 9 ステップ
13 → 9 ステップ
14 → 17 ステップ
15 → 17 ステップ
16 → 4 ステップ
17 → 12 ステップ
18 → 20 ステップ
19 → 20 ステップ
20 → 7 ステップ
21 → 7 ステップ
22 → 15 ステップ
23 → 15 ステップ
24 → 10 ステップ
25 → 23 ステップ
26 → 10 ステップ
27 → 111 ステップ
28 → 18 ステップ
29 → 18 ステップ
30 → 18 ステップ


In [37]:
# 1〜10000で一番長い旅をする数は?(sapplyで一括適用)
steps <- sapply(1:10000, collatz_steps)
cat(which.max(steps), "が最長で", max(steps), "ステップ!\n")

6171 が最長で 261 ステップ!


## Day 2 の演習問題

### 問題1: 496が完全数であることを確かめよう
### 問題2: 2桁の素数の個数/200以下の双子素数
### 問題3: 好きな数でコラッツの旅、1〜100で最長の数
### 問題4: カプレカ数(6174)の実験

## 解答欄

In [38]:
# 問題1



In [39]:
# 問題2



In [40]:
# 問題3



In [41]:
# 問題4



## Day 2 まとめ

- `(1:n)[条件]` —— ベクトルの絞り込みで約数も素数も「全部探す」
- gcdは自作(`while`文の練習にぴったり)
- 大きな整数は gmp パッケージ
- `sapply` + `which.max` で「一番○○な数」を一発検索!